# DACN Eval — Real LLM on Colab T4

**Runtime**: GPU T4 (Runtime > Change runtime type > T4 GPU)

Pipeline: Install Ollama → Pull model → Clone repo → Run dispatcher on fixture

In [ ]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 2: Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print('Ollama server started, PID:', proc.pid)

In [ ]:
# Cell 3: Pull model (chon 1 trong cac model ben duoi)
# gemma4:e2b  — 5.1B, tool-calling, 7.2GB download
# qwen2.5:3b  — 3B, tool-calling, 1.9GB download (nhanh nhat)
# qwen2.5:7b  — 7B, tool-calling, 4.7GB download

MODEL = 'qwen2.5:3b'  # ← doi model o day

!ollama pull {MODEL}
!ollama list

In [ ]:
# Cell 4: Quick smoke test — model co chay khong?
import requests, json

r = requests.post('http://localhost:11434/api/generate', json={
    'model': MODEL,
    'prompt': 'Reply with the single word OK.',
    'stream': False,
    'options': {'temperature': 0}
})
data = r.json()
print(f"Response: {data['response'][:200]}")
print(f"Tokens: prompt={data.get('prompt_eval_count', '?')}, "
      f"completion={data.get('eval_count', '?')}, "
      f"duration_ms={data.get('total_duration', 0) // 1_000_000}")

In [ ]:
# Cell 5: Clone repo + install
!git clone https://github.com/NoSpaceAvailable/DACN.git /content/DACN 2>/dev/null || (cd /content/DACN && git pull)
%cd /content/DACN
!pip install -e . -q
!pip install z3-solver -q
print('\nInstalled OK')

In [ ]:
# Cell 6: Run single fixture through dispatcher with real LLM
import os
os.environ['OLLAMA_BASE_URL'] = 'http://localhost:11434'
os.environ['LLM_BACKEND_DISPATCHER'] = f'ollama:{MODEL}'

FIXTURE = 'data/fixtures/challenge_idor_01'

!python -m vapt_orchestrator_safe.cli dispatch \
    --fixture {FIXTURE} \
    --llm ollama:{MODEL} \
    --max-steps 15

In [ ]:
# Cell 7: Xem ket qua
import glob, json

# Tim run_summary moi nhat
summaries = sorted(glob.glob('outputs/*/run_summary.json'))
if summaries:
    latest = summaries[-1]
    with open(latest) as f:
        summary = json.load(f)
    print(f"Fixture:    {summary['fixture_id']}")
    print(f"Status:     {summary['status']}")
    print(f"Backend:    {summary['backend']}")
    print(f"Steps:      {summary['steps']} (stop: {summary['stop_reason']})")
    print(f"Tool calls: {len(summary['tool_invocations'])}")
    print(f"Findings:   {len(summary.get('validated_findings', []))}")
    print(f"\nTool sequence:")
    for inv in summary['tool_invocations']:
        print(f"  {inv['name']}")
else:
    print('No run_summary.json found. Check errors above.')

In [ ]:
# Cell 8 (optional): Run ALL 3 fixtures
fixtures = [
    'data/fixtures/challenge_idor_01',
    'data/fixtures/challenge_ssrf_01',
    'data/fixtures/challenge_sqli_01',
]

for fix in fixtures:
    print(f'\n{"="*60}')
    print(f'Running: {fix}')
    print('='*60)
    !python -m vapt_orchestrator_safe.cli dispatch \
        --fixture {fix} \
        --llm ollama:{MODEL} \
        --max-steps 15